# Discovery + Silver: `billing.invoices`

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.billing__invoices", engine)
df.shape

(50000, 10)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

invoice_id              object
issued_at               object
due_at                  object
total                   object
status                  object
currency                object
customer_id             object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,invoice_id,issued_at,due_at,total,status,currency,customer_id,_source_file,_ingested_at,_dag_run_id
0,INV-00000001,2025-08-18,2025-09-05,107.64,paid,CLP,CUS-0000003,billing/invoices.csv,2026-07-21 15:02:03.973385,manual__2026-07-21T15:01:58.988673+00:00
1,INV-00000002,2024-05-14,2024-06-04,70.06,paid,EUR,CUS-0000281,billing/invoices.csv,2026-07-21 15:02:03.973385,manual__2026-07-21T15:01:58.988673+00:00
2,INV-00000003,2025-07-23,2025-08-31,35.12,paid,PEN,CUS-0001990,billing/invoices.csv,2026-07-21 15:02:03.973385,manual__2026-07-21T15:01:58.988673+00:00
3,INV-00000004,2023-06-09,2023-06-18,16.19,paid,COP,CUS-0006989,billing/invoices.csv,2026-07-21 15:02:03.973385,manual__2026-07-21T15:01:58.988673+00:00
4,INV-00000005,2023-07-18,2023-09-01,27.62,paid,EUR,CUS-0000366,billing/invoices.csv,2026-07-21 15:02:03.973385,manual__2026-07-21T15:01:58.988673+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("invoice_id duplicados:", df["invoice_id"].duplicated().sum())

customers = pd.read_sql("SELECT customer_id FROM silver.billing__customers", engine)
print("customer_id huerfanos:", (~df["customer_id"].isin(customers["customer_id"])).sum())

Nulos por columna:
invoice_id      0
issued_at       0
due_at          0
total           0
status          0
currency        0
customer_id     0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

invoice_id duplicados: 0
customer_id huerfanos: 0


## 3. `status`, `currency`, `total` y consistencia de fechas

In [4]:
print("status:")
print(df["status"].value_counts())
print()
print("currency:")
print(df["currency"].value_counts())
print()

total = pd.to_numeric(df["total"], errors="coerce")
print("total <= 0:", (total <= 0).sum())

issued = pd.to_datetime(df["issued_at"])
due = pd.to_datetime(df["due_at"])
print("due_at < issued_at (imposible):", (due < issued).sum())

status:
status
paid       34966
pending    10048
overdue     4986
Name: count, dtype: int64

currency:
currency
USD    15162
CLP    15099
EUR     5000
MXN     4876
COP     2540
ARS     2456
PEN     2455
BRL     2412
Name: count, dtype: int64

total <= 0: 0
due_at < issued_at (imposible): 0


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, 0 FKs huerfanas, `total` siempre positivo, `due_at` nunca antes de `issued_at`). Solo tipado y estandarizacion.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["invoice_id", "customer_id", "issued_at", "due_at", "total", "status", "currency"]].copy()

df_silver["issued_at"] = pd.to_datetime(df_silver["issued_at"]).dt.date
df_silver["due_at"] = pd.to_datetime(df_silver["due_at"]).dt.date
df_silver["total"] = pd.to_numeric(df_silver["total"], errors="raise")
df_silver["status"] = df_silver["status"].str.strip().str.lower()
df_silver["currency"] = df_silver["currency"].str.strip().str.upper()

df_silver.head()

,invoice_id,customer_id,issued_at,due_at,total,status,currency
0,INV-00000001,CUS-0000003,2025-08-18,2025-09-05,107.64,paid,CLP
1,INV-00000002,CUS-0000281,2024-05-14,2024-06-04,70.06,paid,EUR
2,INV-00000003,CUS-0001990,2025-07-23,2025-08-31,35.12,paid,PEN
3,INV-00000004,CUS-0006989,2023-06-09,2023-06-18,16.19,paid,COP
4,INV-00000005,CUS-0000366,2023-07-18,2023-09-01,27.62,paid,EUR


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["invoice_id"].is_unique
assert df_silver["customer_id"].isin(customers["customer_id"]).all()
assert df_silver["total"].gt(0).all()
assert (df_silver["due_at"] >= df_silver["issued_at"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 50000 filas listas para silver


## 7. Escribir en `silver.billing__invoices`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "billing.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.billing__invoices CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "billing__invoices",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)
print("Escrito en silver.billing__invoices")

OK: billing.sql ejecutado


Escrito en silver.billing__invoices


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.billing__invoices LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT invoice_id) AS ids_unicos FROM silver.billing__invoices", engine))
check

   filas  ids_unicos
0  50000       50000


,invoice_id,customer_id,issued_at,due_at,total,status,currency,_silver_loaded_at
0,INV-00000001,CUS-0000003,2025-08-18,2025-09-05,107.64,paid,CLP,2026-07-21 15:03:07.317870+00:00
1,INV-00000002,CUS-0000281,2024-05-14,2024-06-04,70.06,paid,EUR,2026-07-21 15:03:07.317870+00:00
2,INV-00000003,CUS-0001990,2025-07-23,2025-08-31,35.12,paid,PEN,2026-07-21 15:03:07.317870+00:00
3,INV-00000004,CUS-0006989,2023-06-09,2023-06-18,16.19,paid,COP,2026-07-21 15:03:07.317870+00:00
4,INV-00000005,CUS-0000366,2023-07-18,2023-09-01,27.62,paid,EUR,2026-07-21 15:03:07.317870+00:00
